### Setup (always run)


In [ ]:
import sys
import subprocess

def install_dependencies():
    packages = ["PyYAML", "fabrictestbed-extensions"]
    print("Checking/Installing deployment dependencies...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
        print("Deployment dependencies installed.\n")
    except Exception as e:
        print(f"Failed to install dependencies: {e}")
        sys.exit(1)

install_dependencies()

from src.util.slice_manager import SliceManager as slice_manager
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager(project_id="f5b4fc7d-978f-45be-9530-b38db8ef5046");
fablib.verify_and_configure();

sm = slice_manager(fablib)

### Deploy on FABRIC

In [ ]:
sm.deploy()

In [ ]:
import time

time.sleep(10)
sm.setup_nodes()

##### Run to update the source code at your nodes, if you make changes

In [ ]:
sm.deploy()
sm.upload_src_files_to_nodes()

### Monte Carlo Test

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import time
import math
from src.schema import Schema
import pandas as pd

sm.deploy() # Check that slice exists
sm.run()

progress = widgets.FloatProgress(value=0.0, min=0.0, max=1.0, description='Accuracy:')
label = widgets.Label(value="Pi Estimate: Calculating...")
display(progress, label)

samples = 0
monte_carlo = Schema(worker_id=0, hits=0, total=0)
previous_len = 0

sm.download_sink_from_receiver("sink.csv")

def update_ui(current_pi, total_samples):
    global previous_len
    df = pd.read_csv('sink.csv')
    
    if len(df) > previous_len:
        new_rows = df.iloc[previous_len:]
        #print(f"New data added ({len(new_rows)} rows):")
        #display(new_rows)
    previous_len = len(df)
    
    monte_carlo.hits = df.tail(1)['hits'].values[0] if not df.empty else 0
    monte_carlo.total = df.tail(1)['total'].values[0] if not df.empty else 0
    error = abs(math.pi - current_pi)
    accuracy = max(0, 1 - error) 
    
    progress.value = accuracy
    label.value = f"Current Pi: {current_pi:.8f} | Samples: {total_samples:,}"

for i in range(1, 2000):
    time.sleep(1)
    pi_estimate = 4 * (monte_carlo.hits / monte_carlo.total) if monte_carlo.total > 0 else 0
    samples = samples + monte_carlo.total
    update_ui(pi_estimate, samples)
    
sm.stop_on_nodes()